In [2]:
# =============================================================================
# IMDB Movie Reviews - Sentiment Analysis (Duygu Analizi) Pipeline
# =============================================================================
# Veri Seti Kaynağı:
#   https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
#
# Gereksinimler:
#   pip install pandas numpy scikit-learn nltk seaborn matplotlib beautifulsoup4
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# Kütüphane İmportları
# ─────────────────────────────────────────────────────────────────────────────
import os
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd

# Metin ön işleme kütüphaneleri
import nltk
from bs4 import BeautifulSoup

# Scikit-learn kütüphaneleri
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Görselleştirme kütüphaneleri
import matplotlib
matplotlib.use("Agg")  # GUI olmayan ortamlar için backend
import matplotlib.pyplot as plt
import seaborn as sns

# Uyarıları bastır (çıktı temizliği için)
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────────────
# NLTK Kaynaklarının İndirilmesi
# ─────────────────────────────────────────────────────────────────────────────
def download_nltk_resources():
    """Gerekli NLTK kaynaklarını kontrol edip indirir."""
    resources = ["stopwords", "punkt", "punkt_tab"]
    for resource in resources:
        try:
            nltk.data.find(f"corpora/{resource}" if resource == "stopwords" else f"tokenizers/{resource}")
        except LookupError:
            print(f"[INFO] NLTK kaynağı indiriliyor: {resource}")
            nltk.download(resource, quiet=True)


download_nltk_resources()

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Stopwords setini bir kere yükle (performans için)
STOP_WORDS = set(stopwords.words("english"))


# =============================================================================
# YARDIMCI FONKSİYONLAR
# =============================================================================

def print_header(title: str) -> None:
    """Bölüm başlığını profesyonel formatta yazdırır."""
    line = "═" * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(f"{line}")


def print_subheader(title: str) -> None:
    """Alt başlık yazdırır."""
    print(f"\n── {title} {'─' * (65 - len(title))}")


# =============================================================================
# STEP 1: VERİ YÜKLEME VE HAZIRLAMA (Loading & Splitting)
# =============================================================================

def load_and_prepare_data(filepath: str) -> tuple:
    """
    Veri setini yükler, label encoding uygular ve eğitim/test olarak böler.

    Parameters
    ----------
    filepath : str
        CSV dosyasının yolu.

    Returns
    -------
    tuple
        (X_train, X_test, y_train, y_test, df) - bölünmüş veri ve orijinal DataFrame
    """
    print_header("STEP 1: VERİ YÜKLEME VE HAZIRLAMA")

    # --- 1.1 Veri yükleme ---
    try:
        df = pd.read_csv(filepath)
        print(f"[✓] Veri seti başarıyla yüklendi: {filepath}")
    except FileNotFoundError:
        print(f"[✗] HATA: '{filepath}' dosyası bulunamadı!")
        print("    Lütfen IMDB veri setini indirip proje dizinine yerleştirin.")
        print("    İndirme bağlantısı: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
        sys.exit(1)
    except Exception as e:
        print(f"[✗] Veri yükleme hatası: {e}")
        sys.exit(1)

    # --- 1.2 Temel veri seti bilgileri ---
    print_subheader("Veri Seti Genel Bilgileri")
    print(f"  Toplam yorum sayısı : {len(df):,}")
    print(f"  Sütunlar            : {list(df.columns)}")
    print(f"  Eksik değer sayısı  : {df.isnull().sum().sum()}")

    # --- 1.3 Sentiment encoding (positive → 1, negative → 0) ---
    df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})

    # Encoding sonrası kontrol
    if df["sentiment"].isnull().any():
        print("[!] UYARI: Beklenmeyen sentiment değerleri tespit edildi. Temizleniyor...")
        df = df.dropna(subset=["sentiment"])
    df["sentiment"] = df["sentiment"].astype(int)

    print_subheader("Sınıf Dağılımı (Encoding Sonrası)")
    class_counts = df["sentiment"].value_counts()
    total = len(df)
    for label, count in class_counts.items():
        label_name = "Positive (1)" if label == 1 else "Negative (0)"
        pct = count / total * 100
        bar = "█" * int(pct / 2)
        print(f"  {label_name}: {count:>6,}  ({pct:.1f}%)  {bar}")

    balance_ratio = class_counts.min() / class_counts.max()
    if balance_ratio > 0.9:
        print(f"\n  → Veri seti DENGELİ (oran: {balance_ratio:.3f})")
    else:
        print(f"\n  → Veri seti DENGESİZ (oran: {balance_ratio:.3f})")

    # --- 1.4 Eğitim / Test bölme (%80 / %20) ---
    X = df["review"]
    y = df["sentiment"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y  # Sınıf dağılımını korur
    )

    print_subheader("Eğitim / Test Bölünmesi")
    print(f"  Eğitim seti boyutu : {len(X_train):,} (%80)")
    print(f"  Test seti boyutu   : {len(X_test):,}  (%20)")

    return X_train, X_test, y_train, y_test, df


# =============================================================================
# STEP 2: GELİŞMİŞ METİN ÖN İŞLEME (Text Preprocessing)
# =============================================================================

def preprocess_text(text: str) -> str:
    """
    Tek bir metin üzerinde kapsamlı ön işleme uygular.

    İşlem Adımları:
        1. Küçük harfe çevirme
        2. HTML etiketlerini temizleme (BeautifulSoup)
        3. URL'leri kaldırma
        4. Noktalama, özel karakter ve sayıları kaldırma
        5. Tokenizasyon (NLTK word_tokenize)
        6. Stopwords filtrelemesi
        7. Kısa kelimeleri (1 karakter) kaldırma

    Parameters
    ----------
    text : str
        Ham yorum metni.

    Returns
    -------
    str
        Temizlenmiş ve işlenmiş metin.
    """
    # Güvenlik kontrolü
    if not isinstance(text, str):
        return ""

    # 1. Küçük harfe çevir
    text = text.lower()

    # 2. HTML etiketlerini temizle
    text = BeautifulSoup(text, "html.parser").get_text()

    # 3. URL'leri kaldır
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # 4. Noktalama, özel karakter ve sayıları kaldır (sadece harfler ve boşluk kalsın)
    text = re.sub(r"[^a-z\s]", "", text)

    # 5. Fazla boşlukları temizle
    text = re.sub(r"\s+", " ", text).strip()

    # 6. Tokenizasyon
    tokens = word_tokenize(text)

    # 7. Stopwords filtrelemesi + kısa kelime (1 karakter) kaldırma
    tokens = [word for word in tokens if word not in STOP_WORDS and len(word) > 1]

    return " ".join(tokens)


def apply_preprocessing(X_train: pd.Series, X_test: pd.Series) -> tuple:
    """
    Eğitim ve test verilerine ön işleme uygular.

    Parameters
    ----------
    X_train : pd.Series
        Eğitim yorum metinleri.
    X_test : pd.Series
        Test yorum metinleri.

    Returns
    -------
    tuple
        (X_train_clean, X_test_clean) - Temizlenmiş metin serileri.
    """
    print_header("STEP 2: METİN ÖN İŞLEME")
    print("[...] Ön işleme uygulanıyor (bu adım birkaç dakika sürebilir)...")

    start_time = time.time()

    X_train_clean = X_train.apply(preprocess_text)
    X_test_clean = X_test.apply(preprocess_text)

    elapsed = time.time() - start_time

    print(f"[✓] Ön işleme tamamlandı! (Süre: {elapsed:.1f} saniye)")

    # Ön işleme öncesi / sonrası karşılaştırma örneği
    print_subheader("Ön İşleme Örneği (İlk Yorum)")
    sample_idx = X_train.index[0]
    print(f"  HAM METİN (ilk 150 karakter):")
    print(f"    \"{X_train.iloc[0][:150]}...\"")
    print(f"\n  TEMİZLENMİŞ METİN (ilk 150 karakter):")
    print(f"    \"{X_train_clean.iloc[0][:150]}...\"")

    return X_train_clean, X_test_clean


# =============================================================================
# STEP 3: VEKTÖRLEŞTİRME (Feature Extraction - TF-IDF)
# =============================================================================

def vectorize_text(X_train_clean: pd.Series, X_test_clean: pd.Series) -> tuple:
    """
    TF-IDF vektörleştirme uygular.

    Parametreler:
        - ngram_range=(1, 2): Unigram ve bigram'ları yakalar.
        - max_features=5000: Bellek yönetimi için özellik sayısını sınırlar.

    Parameters
    ----------
    X_train_clean : pd.Series
        Temizlenmiş eğitim metinleri.
    X_test_clean : pd.Series
        Temizlenmiş test metinleri.

    Returns
    -------
    tuple
        (X_train_tfidf, X_test_tfidf, vectorizer)
    """
    print_header("STEP 3: TF-IDF VEKTÖRLEŞTİRME")

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),     # Unigram + Bigram
        max_features=5000,      # En önemli 5000 özellik
        sublinear_tf=True,      # TF değerlerine logaritmik ölçekleme
        min_df=2,               # En az 2 dokümanda geçen kelimeler
    )

    print("[...] TF-IDF dönüşümü uygulanıyor...")
    start_time = time.time()

    X_train_tfidf = vectorizer.fit_transform(X_train_clean)
    X_test_tfidf = vectorizer.transform(X_test_clean)

    elapsed = time.time() - start_time

    print(f"[✓] TF-IDF dönüşümü tamamlandı! (Süre: {elapsed:.2f} saniye)")
    print(f"\n  Eğitim matrisi boyutu : {X_train_tfidf.shape}")
    print(f"  Test matrisi boyutu   : {X_test_tfidf.shape}")
    print(f"  Toplam özellik sayısı : {len(vectorizer.get_feature_names_out()):,}")

    # En yüksek IDF değerine sahip kelimeler
    print_subheader("Örnek TF-IDF Özellikleri (İlk 20 Kelime)")
    feature_names = vectorizer.get_feature_names_out()
    idf_scores = vectorizer.idf_
    top_indices = np.argsort(idf_scores)[:20]
    for i, idx in enumerate(top_indices, 1):
        print(f"  {i:>2}. {feature_names[idx]:<25} (IDF: {idf_scores[idx]:.4f})")

    return X_train_tfidf, X_test_tfidf, vectorizer


# =============================================================================
# STEP 4: MODEL SEÇİMİ VE EĞİTİMİ (Logistic Regression)
# =============================================================================

def train_model(X_train_tfidf, y_train) -> LogisticRegression:
    """
    Logistic Regression modelini eğitir.

    Metin sınıflandırma görevleri için Logistic Regression tercih edilmiştir:
      - Yüksek boyutlu seyrek veriler üzerinde iyi performans gösterir.
      - Eğitim süresi kısa ve yorumlanabilirliği yüksektir.

    Parameters
    ----------
    X_train_tfidf : sparse matrix
        TF-IDF eğitim matrisi.
    y_train : pd.Series
        Eğitim etiketleri.

    Returns
    -------
    LogisticRegression
        Eğitilmiş model.
    """
    print_header("STEP 4: MODEL EĞİTİMİ (Logistic Regression)")

    model = LogisticRegression(
        max_iter=1000,       # Yakınsama için yeterli iterasyon
        solver="lbfgs",      # Büyük veri setleri için uygun çözücü
        C=1.0,               # Düzenlileştirme parametresi
        random_state=42,     # Tekrarlanabilirlik
        n_jobs=-1,           # Tüm CPU çekirdeklerini kullan
    )

    print("[...] Model eğitiliyor...")
    start_time = time.time()

    model.fit(X_train_tfidf, y_train)

    training_time = time.time() - start_time

    print(f"[✓] Model eğitimi tamamlandı!")
    print(f"\n  Model           : Logistic Regression")
    print(f"  Solver          : {model.solver}")
    print(f"  Max Iterations  : {model.max_iter}")
    print(f"  C (Reg. Param)  : {model.C}")
    print(f"  Eğitim Süresi   : {training_time:.2f} saniye")

    return model


# =============================================================================
# STEP 5: PERFORMANS DEĞERLENDİRME (Evaluation & Reporting)
# =============================================================================

def evaluate_model(
    model: LogisticRegression,
    X_test_tfidf,
    y_test: pd.Series,
    X_test_original: pd.Series,
) -> None:
    """
    Model performansını kapsamlı şekilde değerlendirir ve raporlar.

    Raporlanan Metrikler:
        - Accuracy Score
        - Classification Report (Precision, Recall, F1-Score)
        - Confusion Matrix (Görselleştirme)
        - 5 Örnek Karşılaştırma Tablosu

    Parameters
    ----------
    model : LogisticRegression
        Eğitilmiş model.
    X_test_tfidf : sparse matrix
        TF-IDF test matrisi.
    y_test : pd.Series
        Gerçek test etiketleri.
    X_test_original : pd.Series
        Orijinal (temizlenmemiş) test metinleri.
    """
    print_header("STEP 5: PERFORMANS DEĞERLENDİRME")

    # --- 5.1 Tahmin ---
    y_pred = model.predict(X_test_tfidf)

    # --- 5.2 Accuracy Score ---
    accuracy = accuracy_score(y_test, y_pred)
    print_subheader("Accuracy (Doğruluk Oranı)")
    bar_len = int(accuracy * 40)
    bar = "█" * bar_len + "░" * (40 - bar_len)
    print(f"  Accuracy: {accuracy:.4f}  ({accuracy * 100:.2f}%)")
    print(f"  [{bar}]")

    # --- 5.3 Classification Report ---
    print_subheader("Classification Report (Sınıflandırma Raporu)")
    target_names = ["Negative (0)", "Positive (1)"]
    report = classification_report(y_test, y_pred, target_names=target_names)
    print(report)

    # --- 5.4 Confusion Matrix ---
    print_subheader("Confusion Matrix (Karışıklık Matrisi)")
    cm = confusion_matrix(y_test, y_pred)

    # Terminal çıktısı olarak matris
    print(f"\n  {'':>18} {'Tahmin: Neg':>14} {'Tahmin: Pos':>14}")
    print(f"  {'Gerçek: Neg':>18} {cm[0][0]:>14,} {cm[0][1]:>14,}")
    print(f"  {'Gerçek: Pos':>18} {cm[1][0]:>14,} {cm[1][1]:>14,}")

    # Seaborn ile görselleştirme
    try:
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt=",d",
            cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"],
            linewidths=1,
            linecolor="white",
            cbar_kws={"label": "Tahmin Sayısı"},
        )
        plt.title("Confusion Matrix - IMDB Sentiment Analysis", fontsize=14, fontweight="bold", pad=15)
        plt.xlabel("Tahmin Edilen Sınıf", fontsize=12)
        plt.ylabel("Gerçek Sınıf", fontsize=12)
        plt.tight_layout()

        # Kaydet
        output_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "confusion_matrix.png")
        plt.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"\n  [✓] Confusion Matrix görseli kaydedildi: {output_path}")
    except Exception as e:
        print(f"\n  [!] Görselleştirme kaydedilemedi: {e}")

    # --- 5.5 Örnek Karşılaştırma Tablosu (5 Örnek) ---
    print_subheader("Örnek Tahmin Karşılaştırması (5 Yorum)")

    # İndeksleri sıfırla ve rastgele 5 örnek seç
    test_indices = y_test.index.tolist()
    np.random.seed(42)
    sample_indices = np.random.choice(len(test_indices), size=5, replace=False)

    sentiment_map = {0: "Negative 😞", 1: "Positive 😊"}

    print(f"\n  {'No':<4} {'Gerçek Duygu':<16} {'Tahmin Duygu':<16} {'Sonuç':<8} Yorum (ilk 70 karakter)")
    print(f"  {'─' * 4} {'─' * 16} {'─' * 16} {'─' * 8} {'─' * 45}")

    for i, idx in enumerate(sample_indices, 1):
        real_idx = test_indices[idx]
        actual = y_test.iloc[idx]
        predicted = y_pred[idx]
        review_text = X_test_original.iloc[idx]

        # Yorum metnini kısalt
        short_review = review_text[:70].replace("\n", " ").replace("<br />", " ")
        if len(review_text) > 70:
            short_review += "..."

        match_icon = "  ✓" if actual == predicted else "  ✗"

        print(
            f"  {i:<4} "
            f"{sentiment_map[actual]:<16} "
            f"{sentiment_map[predicted]:<16} "
            f"{match_icon:<8} "
            f"{short_review}"
        )

    # --- 5.6 Genel Özet ---
    correct = (y_test.values == y_pred).sum()
    incorrect = len(y_test) - correct

    print_header("GENEL SONUÇ ÖZETİ")
    print(f"  Model              : Logistic Regression")
    print(f"  Vektörleştirme     : TF-IDF (unigram + bigram, max_features=5000)")
    print(f"  Test Seti Boyutu   : {len(y_test):,}")
    print(f"  Doğru Tahmin       : {correct:,}")
    print(f"  Yanlış Tahmin      : {incorrect:,}")
    print(f"  Accuracy           : {accuracy:.4f} ({accuracy * 100:.2f}%)")
    print(f"\n  {'═' * 70}")


# =============================================================================
# ANA PROGRAM (Main)
# =============================================================================

def main():
    """Pipeline'ın ana çalıştırma fonksiyonu."""

    print("\n" + "╔" + "═" * 70 + "╗")
    print("║" + " IMDB 50K Movie Reviews - Sentiment Analysis Pipeline ".center(70) + "║")
    print("║" + " Logistic Regression + TF-IDF ".center(70) + "║")
    print("╚" + "═" * 70 + "╝")

    # Veri seti dosya yolu
    # Kullanıcı CSV dosyasını bu script ile aynı dizine yerleştirmelidir
    #script_dir = os.path.dirname(os.path.abspath(__file__))
    #dataset_path = os.path.join(script_dir, "IMDB Dataset.csv")

    # Colab'in varsayılan çalışma dizini
    script_dir = "/content"
    dataset_path = os.path.join(script_dir, "IMDB Dataset.csv")

    # Alternatif dosya adları kontrolü
    if not os.path.exists(dataset_path):
        alternatives = [
            "IMDB_Dataset.csv",
            "imdb_dataset.csv",
            "IMDB Dataset.csv",
            "imdb-dataset.csv",
            "IMDB.csv",
        ]
        for alt in alternatives:
            alt_path = os.path.join(script_dir, alt)
            if os.path.exists(alt_path):
                dataset_path = alt_path
                break

    pipeline_start = time.time()

    # ── Step 1: Veri Yükleme ve Hazırlama ──
    X_train, X_test, y_train, y_test, df = load_and_prepare_data(dataset_path)

    # ── Step 2: Metin Ön İşleme ──
    X_train_clean, X_test_clean = apply_preprocessing(X_train, X_test)

    # ── Step 3: TF-IDF Vektörleştirme ──
    X_train_tfidf, X_test_tfidf, vectorizer = vectorize_text(X_train_clean, X_test_clean)

    # ── Step 4: Model Eğitimi ──
    model = train_model(X_train_tfidf, y_train)

    # ── Step 5: Performans Değerlendirme ──
    evaluate_model(model, X_test_tfidf, y_test, X_test)

    # Toplam çalışma süresi
    total_time = time.time() - pipeline_start
    print(f"\n  ⏱  Toplam Pipeline Süresi: {total_time:.1f} saniye ({total_time / 60:.1f} dakika)")
    print(f"\n{'═' * 72}\n")


# ─────────────────────────────────────────────────────────────────────────────
# Script doğrudan çalıştırıldığında main() fonksiyonunu başlat
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════════════════╗
║         IMDB 50K Movie Reviews - Sentiment Analysis Pipeline         ║
║                     Logistic Regression + TF-IDF                     ║
╚══════════════════════════════════════════════════════════════════════╝

══════════════════════════════════════════════════════════════════════
  STEP 1: VERİ YÜKLEME VE HAZIRLAMA
══════════════════════════════════════════════════════════════════════
[✓] Veri seti başarıyla yüklendi: /content/IMDB Dataset.csv

── Veri Seti Genel Bilgileri ────────────────────────────────────────
  Toplam yorum sayısı : 50,000
  Sütunlar            : ['review', 'sentiment']
  Eksik değer sayısı  : 0

── Sınıf Dağılımı (Encoding Sonrası) ────────────────────────────────
  Positive (1): 25,000  (50.0%)  █████████████████████████
  Negative (0): 25,000  (50.0%)  █████████████████████████

  → Veri seti DENGELİ (oran: 1.000)

── Eğitim / Test Bölünmesi ──────────────────────────────────────